# Expfit notes 4: Confidence intervals

We'd like to set confidence intervals, particularly on obtained time constants.

## Using a profile log-likelihood method

So far, we optimised the mean squared error (MSE) to find a best fit

\begin{align}
\hat{E} = E(\hat{\theta}) = \frac{1}{n}\sum(v_i - f(\hat{\theta})_i)^2
\end{align}

where $\theta$ is a parameter vector of size $m$, $v_i$ is the data, and $f(\theta)$ is the model (a sum of exponentials).
For a particular parameter in the vector, the parameter of interest, we can find a confidence interval (CI) as follows:

1. Fix the parameter of interest at a distance $d > 0$ from its original value
2. Formulate a new optimisation problem over the remaining $m - 1$ parameters, yielding a "test" vector $\theta_t$ and associated $E_t = E(\theta_t)$
3. While $E_t$ is below some acceptable level, increase $d$ and repeat.
4. When an upper bound for $d$ has been found ($E_t$ has exceed the acceptable level), start a bisection search to find the value $d$ for which $E_t$ is exactly at the cut-off point

This gives an upper bound.
The procedure is repeated with $d < 0$ to find the lower bound.

This requires repeated reoptimisation, but for these problems it should still be very fast.

For the "acceptable level" of error, we can intuitively define some cut-off
\begin{align}
E_\text{max} = (1 + c) \hat{E}
\end{align}
where $c$ is a small positive number.

But we can also arrive at this using a statistical approach, which gives a slightly more reasoned approach for choosing $c$.

### The cut-off

First, we'll assume the model $f$ is a sum of exponentials _plus a noise variable distributed as $N(0, \sigma^2)$ for some unknown $\sigma$ - even though this may not be true of our data.

Then, we can write out a likelihood of observing $v$ as
\begin{align}
L(\theta, \sigma | v) \equiv p(v | \theta, \sigma)
 &= \prod^n_{i=1} \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left( - \frac{(v_i - f(\theta)_i)^2}{2\sigma^2} \right)
\end{align}
leading to a log-likelihood of
\begin{align}
\mathcal{l}(\theta, \sigma | v) \equiv \log L(\theta, \sigma | v)
 &= -\frac{n}{2}\log(2\pi) - n\log(\sigma) - \frac{1}{2\sigma^2} \sum^n_{i=1} (v_i - f(\theta)_i)^2
\end{align}

Now we want to compare the likelihood of our test value $\theta_t$ with the likelihood of the optimum $\hat{\theta}$.
This can be done with a [likelihood ratio test](https://en.wikipedia.org/wiki/Likelihood-ratio_test):

\begin{align}
-2 \log \frac{L(\theta_t)}{L(\hat{\theta})} < \chi^2_{1,1-\alpha}
\end{align}

where $\chi_{1,1-\alpha}$ is the value [Chi-squared distribution](https://en.wikipedia.org/wiki/Chi-squared_distribution) with $1$ degree of freedom and $1-\alpha$ is the level of confidence.
For example, if we want a CI that contains the true solution with 95% probability, we use $\chi_{1,0.95} \approx 3.841$.
The "one degree of freedom" here refers to the different parameter vector sizes: $m$ for the original $\hat{\theta}$ and $m - 1$ for the tested value $\theta_t$.
(So if we had fixed two parameters, an extension to this method, we would use the 2 degrees of freedom statistic.)

Values of $\chi^2$ can be found in tables, or obtained from Scipy:

In [1]:
import scipy
print(scipy.stats.chi2.ppf(0.99, 1))
print(scipy.stats.chi2.ppf(0.95, 1))
print(scipy.stats.chi2.ppf(0.90, 1))

6.6348966010212145
3.841458820694124
2.705543454095404


_Note that the "level of confidence" refers to the chances of the true solution being within our interval, not the certainty of our optimum! So increased confidence (the 0.99 value) means a much wider interval._

We can rewrite this inequality to find a cut-off in terms of our chosen $\chi^2$.

First, rewrite as log likelihoods.
\begin{align}
-2 \log\frac{L(\theta_t)}{L(\hat{\theta})} = -2 (\mathcal{l}(\theta_t) - \mathcal{l}(\hat{\theta}))
\end{align}

Next, because $n$ and $\sigma$ are the same for both problems (we're not inferring $\sigma$ here, just assuming it's known):
\begin{align}
-2 \cdot -\frac{1}{2\sigma^2} \left(\sum (v_i - f(\theta_t)_i)^2 - \sum (v_i - f(\hat{\theta})_i)^2\right)
= \frac{n}{\sigma^2} (E(\theta_t) - \hat{E})
\end{align}

Our best estimate for $\sigma^2$ is the variance in the residuals of our best fit
\begin{align}
\sigma^2 \approx \frac{1}{n}\sum(v_i - f(\hat{\theta})_i)^2 = \hat{E}
\end{align}
So we can fill in $\hat{E}$ for $\sigma^2$ to find
\begin{align}
\frac{n}{\sigma^2} (E_t - \hat{E}) \approx n \left(\frac{E_t}{\hat{E}} - 1\right) < \chi^2_{1,1-\alpha}
\end{align}

which would mean our confidence interval includes all values for which

\begin{align}
E_t < (1 + \chi^2_{1,1-\alpha} / n) \hat{E}
\end{align}

Note the appearance of $n$: more data gives tigther bounds.

## Fisher information method

A cheaper, but probably less accurate method uses [Fisher information](https://en.wikipedia.org/wiki/Fisher_information).
This starts from a property of maximum likelihood estimation (MLE) called _asymptotic normality_, which states that near the true solution $\theta_\text{true}$, the _maximum likelihood estimate_ $\hat{\theta}$ has a Normal distribution:
\begin{align}
\hat{\theta} \sim N(\theta_\text{true}, I^{-1}(\theta_\text{true}))
\end{align}
Here we treat $\hat{\theta}$ as a random variable because it's based on randomly sampled data --- resampling the data $v$ with the same $\theta_\text{true}$ would provide a different estimate $\hat{\theta}$.
(We'll assume the optimiser is perfect and deterministic.)

The quantity $I(\theta)$ is called the Fisher information matrix, and is defined as
\begin{align}
I_{ij}(\theta) = \mathbb{E} \left( -\frac{\partial^2\mathcal{l}(\theta|v)}{\partial \theta_i \partial \theta_j} \right)
\end{align}
where $\mathbb{E}()$ is the expected value.
We don't know this, but can approximate as
\begin{align}
I_{ij}(\theta) \approx J_{ij}(\hat{\theta}) = \left. -\frac{\partial^2\mathcal{l}(\theta|v)}{\partial \theta_i \partial \theta_j} \right\vert_{\theta = \hat{\theta}}
\end{align}
leading to an approximate covariance matrix
\begin{align}
\Sigma = J^{-1}
\end{align}

### Reformulation in terms of $E$ and $H$

To use this, we relate the second derivative above to the Hessian of our MSE.

As above, we **assume a Normally distributed additive noise term**, and a **known and fixed $\sigma$ and $n$**.
With these assumptions, taking the second derivative of

\begin{align}
\mathcal{l}(\theta, \sigma | v) = -\frac{n}{2}\log(2\pi) - n\log(\sigma) - \frac{1}{2\sigma^2} \sum^n_{i=1} (v_i - f(\theta)_i)^2
\end{align}
the first terms disappear for
\begin{align}
\frac{\partial^2}{\partial \theta^2}\mathcal{l}(\theta) = -\frac{1}{2\sigma^2} \frac{\partial^2}{\partial \theta^2} \sum^n_{i=1} (v_i - f(\theta)_i)^2
\end{align}

While for $E$ we find
\begin{align}
\frac{\partial^2}{\partial \theta^2} E = \frac{1}{n} \frac{\partial^2}{\partial \theta^2} \sum(v_i - f(\hat{\theta})_i)^2 \equiv H(\theta)
\end{align}
and so
\begin{align}
n H(\theta) = -2\sigma^2 \frac{\partial^2}{\partial \theta^2}\mathcal{l}(\theta)
\quad \rightarrow \quad
J(\theta) = -\frac{\partial^2}{\partial \theta^2}\mathcal{l}(\theta) = \frac{n}{2\sigma^2} H(\theta)
\end{align}

Finally, we fill in $\sigma^2 \approx \hat{E}$ for
\begin{align}
J(\theta) \approx \frac{n}{2\hat{E}} H(\theta)
\end{align}

### Using $J$

To use this, we go back to
\begin{align}
\hat{\theta} \sim N(\theta_\text{true}, I^{-1}(\theta_\text{true}))
\end{align}
which says that our best fit value $\hat{\theta}$ should be distributed around the unknown $\theta_\text{true}$ with covariance
\begin{align}
\Sigma = I^{-1}(\theta_\text{true}) \approx J^{-1}(\hat{\theta})
\end{align}
so we can interpret our best fit as a sample around the true solution with a covariance
\begin{align}
\Sigma \approx J^{-1}(\hat{\theta}) \approx \frac{2E(\hat{\theta})}{n} H^{-1}(\theta)
\end{align}

For single-parameter confidence intervals, we can use the diagonal entries $\sigma^2_{ii} = \Sigma_{ii}$ and rewrite
\begin{align}
\hat{\theta} \sim N(\theta_\text{true}, \Sigma)
\quad \rightarrow \quad
\hat{\theta}_i \sim N(\theta_{\text{true},i}, \sigma^2_{ii})
\end{align}
as
\begin{align}
\frac{\hat{\theta} - \theta_\text{true}}{\sigma_{ii}} \sim N(0, 1)
\end{align}
Because this is a symmetric distribution, using the 95% percentile gives a 90% confidence bound:
\begin{align}
CI_{90} = \hat{\theta} \pm 1.645 \sigma_{ii} 
\end{align}
Again, we can use Scipy to find some percentiles:

In [2]:
print(scipy.stats.norm.ppf(0.975))  # 95%
print(scipy.stats.norm.ppf(0.95))  # 90%

1.959963984540054
1.6448536269514722
